In [1]:
import json
from pathlib import Path

data_dir = Path("../../data") 
system_prompt = (Path("../../prompt/json_SYSTEM_prompt.txt")).read_text(encoding="utf-8")

rows = []
for user_file in sorted(data_dir.glob("user_prompt/*.txt")):
    name = user_file.stem  # e.g. "data1"
    report_file = data_dir / "report" / f"{name}.txt"
    json_file = data_dir / "json" / f"{name}.json"

    if not report_file.exists() or not json_file.exists():
        continue

    rows.append({
        "system_prompt": system_prompt,
        "user_prompt": user_file.read_text(encoding="utf-8"),
        "reasoning": report_file.read_text(encoding="utf-8"),
        "output": json_file.read_text(encoding="utf-8"),
    })


In [2]:
msgs = []
for row in rows:
    reasoning = row["reasoning"]
    output = row["output"]
    assistant_response = f"<think>\n{reasoning}\n</think>\n{output}"

    msgs.append({
        "messages": [
            {"role": "system", "content": row["system_prompt"]},
            {"role": "user", "content": row["user_prompt"]},
            {"role": "assistant", "content": assistant_response},
        ]
    })

In [3]:
with open("data/custom-reasoning.json", "w", encoding="utf-8") as f:
    json.dump(msgs, f, ensure_ascii=False, indent=2)

In [7]:
import json
from pathlib import Path

dataset_info_path = Path("/LLaMA-Factory/data/dataset_info.json")  # change if yours is elsewhere

new_dataset_name = "sunny_reasoning"
new_entry = {
    "file_name": "/LLaMA-Factory/data/custom-reasoning.json",
    "formatting": "sharegpt",
    "columns": {"messages": "messages"},
    "tags": {
        "role_tag": "role",
        "content_tag": "content",
        "user_tag": "user",
        "assistant_tag": "assistant",
        "system_tag": "system",
    },
}

# Load existing dataset_info.json
with dataset_info_path.open("r", encoding="utf-8") as f:
    dataset_info = json.load(f)

# Add or update entry
dataset_info[new_dataset_name] = new_entry

# Save back
dataset_info_path.parent.mkdir(parents=True, exist_ok=True)
with dataset_info_path.open("w", encoding="utf-8") as f:
    json.dump(dataset_info, f, ensure_ascii=False, indent=2)

print(f"Updated {dataset_info_path} with key: {new_dataset_name}")

Updated /LLaMA-Factory/data/dataset_info.json with key: sunny_reasoning
